# Data Engineering: Municipality Indicators for Central Switzerland

This notebook builds a table with one row per municipality in Central Switzerland (Lucerne, Uri, Schwyz, Obwalden, Nidwalden, Zug) and a set of socio-economic indicators. The table is later merged with the Flatfox listings in `data_engineering_real_estate`.

## Approach

1. **Select municipalities:** All municipalities in the permanent resident population file (`staendigepop`, status 2025) are assigned to a canton via their BFS number range. Only the six Central Swiss cantons are kept.
2. **Simple BFS indicators:** Every entry in `BFS_INDICATORS` is loaded from `data/demographic_data/` and merged by BFS municipality number. The key is the column name, the value is the indicator ID (end of the file name) and the reference year. With `None`, the most recent year with values for Central Switzerland is used (at most `MAX_YEAR`). To add or remove an indicator, only one line in `BFS_INDICATORS` needs to be changed.
3. **Indicators with special processing:**
   - **Population growth:** change of the permanent resident population from `POPULATION_FROM_YEAR` to `POPULATION_TO_YEAR` in %. Only calculated if both years refer to the same municipality status.
   - **Social assistance rate:** from a separate BFS file. Values withheld for data protection reasons remain missing.
   - **Median income:** taxable income from the federal tax statistics (`data/tax_data/`), metric `INCOME_METRIC`.
   - **Employment per 100 residents:** sum of the three economic sectors divided by the population of the same year. The source data still list Honau separately, so it is merged into Root, which absorbed it.
4. **Check, validate and save:** The notebook shows the reference year of each variable and the number of missing values, checks the result with a few assertions and then saves the table.

## Output

`data/data_processed/gemeinde_indikatoren_zentralschweiz.csv`: one row per municipality with the indicators, followed by metadata columns (reference year and municipality status of each indicator).

Municipal boundaries are not harmonised between years. The municipality status of each indicator is documented in the `_gemeindestand` columns.

In [ ]:
from pathlib import Path
import warnings

import pandas as pd

In [ ]:
# Files and reference years
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
BFS_DIR = DATA_DIR / "demographic_data"


def bfs_file(indicator_id):
    """Find the CSV of an indicator by its ID at the end of the file name."""
    matches = list(BFS_DIR.glob(f"*__{indicator_id}.csv"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Indicator '{indicator_id}' not uniquely found: {matches}"
        )
    return matches[0]


# Simple BFS indicators: add or remove with a single line.
# Column name: (indicator ID, reference year or None = most recent up to MAX_YEAR)
BFS_INDICATORS = {
    "population_density_productive":    ("bevoelkerungsdichtevf", None),
    "household_size_mean":              ("durchschnittlichehau", None),
    "vacancy_rate_pct":                 ("leerwohnungsziffer", None),
    "population_age65plus_pct":         ("anteilpop65", None),
    "new_dwellings_per_1000_residents": ("neuerstelltewohnungenpro", None),
    "single_family_buildings_pct":      ("anteileinfamhaus", None),
    "foreign_population_pct":           ("anteilauslpop", None),
    "net_migration_per_1000_residents": ("wanderungssaldopro", None),
}

POPULATION_FILE = bfs_file("staendigepop")
EMPLOYMENT_FILES = [bfs_file(f"bsws{sector}") for sector in (1, 2, 3)]
SOCIAL_FILE = bfs_file("sozialhilfequot")
INCOME_FILE = DATA_DIR / "tax_data" / "statistik-dbst-np-kennzahlen-ohne-2022.csv"
OUTPUT_FILE = DATA_DIR / "data_processed" / "gemeinde_indikatoren_zentralschweiz.csv"

POPULATION_FROM_YEAR = 2024
POPULATION_TO_YEAR = 2025
SOCIAL_YEAR = 2024
SOCIAL_MUNICIPALITY_STAND = "01.01.2024"
INCOME_YEAR = 2022

# Income metric used so far.
# For the taxable median equivalised income: "median_steinka".
INCOME_METRIC = "median_steink"

# Upper limit for the most recent available year (BFS_INDICATORS and employment).
MAX_YEAR = 2025


# Helper functions
def load_bfs(path):
    """Load a Map Explorer CSV and standardise numeric columns."""
    data = pd.read_csv(
        path, sep=None, engine="python", encoding="utf-8-sig"
    )

    required = {"gemeinde_bfs", "jahr", "wert", "gemeindestand"}
    if not required.issubset(data.columns):
        raise ValueError(
            f"{path.name}: Required columns are {required}. "
            f"Available: {data.columns.tolist()}"
        )

    data["gemeinde_bfs"] = pd.to_numeric(data["gemeinde_bfs"], errors="raise").astype("Int64")
    data["jahr"] = pd.to_numeric(data["jahr"], errors="raise").astype("Int64")
    data["wert"] = pd.to_numeric(data["wert"], errors="coerce")

    return data.dropna(subset=["gemeinde_bfs", "jahr"]).copy()


def select_year(data, year):
    """Select one year with exactly one row per municipality."""
    selected = data.loc[data["jahr"].eq(year)].copy()

    if selected.empty:
        raise ValueError(f"No data available for the year {year}.")

    if selected["gemeinde_bfs"].duplicated().any():
        raise ValueError(
            f"Several rows per municipality in the year {year}. "
            "Please check spatial levels and municipality status dates."
        )

    return selected


def available_years(data, municipality_ids, max_year=MAX_YEAR):
    """Return the years with available values for the selected municipalities."""
    mask = (
        data["jahr"].le(max_year)
        & data["wert"].notna()
        & data["gemeinde_bfs"].isin(municipality_ids)
    )
    return set(data.loc[mask, "jahr"].astype(int))


def merge_indicator(base, data, year, column_name):
    """Add the values, reference year and municipality status of an indicator."""
    selected = select_year(data, year)

    values = selected[
        ["gemeinde_bfs", "wert", "gemeindestand"]
    ].rename(
        columns={
            "wert": column_name,
            "gemeindestand": f"{column_name}_gemeindestand",
        }
    )
    values[f"{column_name}_year"] = year

    return base.merge(
        values,
        on="gemeinde_bfs",
        how="left",
        validate="one_to_one",
    )


# All municipalities in Central Switzerland
population = load_bfs(POPULATION_FILE)
population_current = select_year(population, POPULATION_TO_YEAR)

df_gemeinden = population_current[["gemeinde_bfs", "gemeinde_name", "gemeindestand"]
    ].rename(columns={"gemeindestand": "gemeindeverzeichnis_stand"}).copy()

canton_ranges = {
    "LU": (1000, 1199),
    "UR": (1200, 1299),
    "SZ": (1300, 1399),
    "OW": (1400, 1499),
    "NW": (1500, 1599),
    "ZG": (1700, 1799),
}

df_gemeinden["kanton_geo"] = pd.Series(pd.NA, index=df_gemeinden.index, dtype="string")

for canton, (lower, upper) in canton_ranges.items():
    mask = df_gemeinden["gemeinde_bfs"].between(lower, upper)
    df_gemeinden.loc[mask, "kanton_geo"] = canton

df_gemeinden = (df_gemeinden.dropna(subset=["kanton_geo"]).reset_index(drop=True))

municipality_ids = df_gemeinden["gemeinde_bfs"]
year_summary = []


# Population growth 2024 → 2025
previous = select_year(population, POPULATION_FROM_YEAR)
current = population_current.loc[population_current["gemeinde_bfs"].isin(municipality_ids)]

growth = current[
    ["gemeinde_bfs", "wert", "gemeindestand"]
].merge(
    previous[["gemeinde_bfs", "wert", "gemeindestand"]],
    on="gemeinde_bfs",
    how="left",
    suffixes=("_to", "_from"),
    validate="one_to_one",
)

# Standardise dates before comparing them.
stand_to = pd.to_datetime(growth["gemeindestand_to"], format="%d.%m.%Y", errors="raise")
stand_from = pd.to_datetime(growth["gemeindestand_from"], format="%d.%m.%Y", errors="raise")

different_stands = (
    stand_to.notna()
    & stand_from.notna()
    & stand_to.ne(stand_from)
)

if different_stands.any():
    raise ValueError(
        "Different municipality status dates for population growth."
    )

comparable_stands = stand_to.notna() & stand_from.notna()

growth["population_growth_pct"] = (
    growth["wert_to"]
    .div(growth["wert_from"].where(growth["wert_from"].gt(0)))
    .sub(1)
    .mul(100)
    .where(comparable_stands)
)

growth["population_growth_from_year"] = POPULATION_FROM_YEAR
growth["population_growth_to_year"] = POPULATION_TO_YEAR
growth["population_growth_gemeindestand"] = (growth["gemeindestand_to"].where(comparable_stands))

df_gemeinden = df_gemeinden.merge(
    growth[
        [
            "gemeinde_bfs",
            "population_growth_pct",
            "population_growth_from_year",
            "population_growth_to_year",
            "population_growth_gemeindestand",
        ]
    ],
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({"variable": "population_growth_pct","reference_year": f"{POPULATION_FROM_YEAR} → {POPULATION_TO_YEAR}",})

# Simple BFS indicators
for column_name, (indicator_id, year) in BFS_INDICATORS.items():
    data = load_bfs(bfs_file(indicator_id))

    if year is None:
        years = available_years(data, municipality_ids)
        if not years:
            raise ValueError(f"No usable years for '{indicator_id}'.")
        year = max(years)

    df_gemeinden = merge_indicator(
        df_gemeinden, data, year, column_name
    )

    year_summary.append({
        "variable": column_name,
        "reference_year": str(year),
    })


# Social assistance rate
social = pd.read_csv(
    SOCIAL_FILE,
    sep=";",
    skiprows=2,
    encoding="utf-8-sig",
    na_values=["N/A - Statistikgeheimnis"],
)

social = social[
    [
        "Identifikator",
        f"Sozialhilfequote {SOCIAL_YEAR}",
        f"Sozialhilfequote {SOCIAL_YEAR} - Schätzung (*=ja)",
    ]
].rename(
    columns={
        "Identifikator": "gemeinde_bfs",
        f"Sozialhilfequote {SOCIAL_YEAR}": "social_assistance_rate_pct",
        f"Sozialhilfequote {SOCIAL_YEAR} - Schätzung (*=ja)":
            "social_assistance_estimated",
    }
).copy()

social["gemeinde_bfs"] = pd.to_numeric(social["gemeinde_bfs"], errors="raise").astype("Int64")
social["social_assistance_rate_pct"] = pd.to_numeric(social["social_assistance_rate_pct"], errors="raise")

social["social_assistance_estimated"] = (
    social["social_assistance_estimated"]
    .fillna("")
    .str.strip()
    .eq("*")
    .astype("boolean")
)

social["social_assistance_rate_pct_year"] = SOCIAL_YEAR
social["social_assistance_rate_pct_gemeindestand"] = (SOCIAL_MUNICIPALITY_STAND)

df_gemeinden = df_gemeinden.merge(
    social.dropna(subset=["gemeinde_bfs"]),
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({
    "variable": "social_assistance_rate_pct",
    "reference_year": str(SOCIAL_YEAR),
})


# Median income from the tax file
tax = pd.read_csv(
    INCOME_FILE,
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
)

tax.columns = [
    " ".join(str(column).replace("\ufeff", "").split())
    for column in tax.columns
]

municipality_columns = [
    column for column in tax.columns
    if column.startswith("Gemeindenummer")
]
unit_columns = [
    column for column in tax.columns
    if column.startswith("Einheit ID")
]

if len(municipality_columns) != 1 or len(unit_columns) != 1:
    raise ValueError(
        f"Tax columns not uniquely identified: {tax.columns.tolist()}"
    )

income = tax.loc[
    tax[unit_columns[0]].str.strip().eq("T"),
    [municipality_columns[0], INCOME_METRIC],
].rename(
    columns={
        municipality_columns[0]: "gemeinde_bfs",
        INCOME_METRIC: "income_median_chf",
    }
).copy()

income["gemeinde_bfs"] = pd.to_numeric(income["gemeinde_bfs"], errors="raise").astype("Int64")
income["income_median_chf"] = pd.to_numeric(income["income_median_chf"].str.replace(r"['’\s]", "", regex=True),errors="coerce",)
income["income_year"] = INCOME_YEAR
income["income_metric"] = INCOME_METRIC

df_gemeinden = df_gemeinden.merge(
    income.dropna(subset=["gemeinde_bfs"]),
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({
    "variable": "income_median_chf",
    "reference_year": str(INCOME_YEAR),
})

# Employment per 100 residents
sector_data = [load_bfs(path) for path in EMPLOYMENT_FILES]
common_years = available_years(population, municipality_ids)

for data in sector_data:
    common_years = common_years.intersection(
        available_years(data, municipality_ids)
    )

if not common_years:
    raise ValueError(
        "No common year available for employment and population."
    )

employment_year = max(common_years)

employment = select_year(population, employment_year)[
    ["gemeinde_bfs", "wert", "gemeindestand"]
].rename(
    columns={
        "wert": "population",
        "gemeindestand": "employment_population_gemeindestand",
    }
)

employment = employment.loc[
    employment["gemeinde_bfs"].isin(municipality_ids)
].copy()

sector_columns = []

for sector, data in enumerate(sector_data, start=1):
    values = select_year(data, employment_year)[
        ["gemeinde_bfs", "wert", "gemeindestand"]
    ].copy()

    # This check is defined for the observed source data.
    source_stands = set(values["gemeindestand"].dropna())
    if (
        source_stands != {"01.01.2024"}
        or values["gemeindestand"].isna().any()
    ):
        raise ValueError(
            f"Sector {sector}: Unexpected municipality status date. "
            "The assignment needs to be checked again."
        )

    # In the 2025 municipality status, Root includes the former municipality of Honau.
    if not {1057, 1065}.issubset(set(values["gemeinde_bfs"])):
        raise ValueError(
            f"Sector {sector}: Honau or Root is missing in the source data."
        )

    values["gemeinde_bfs"] = values["gemeinde_bfs"].replace({1057: 1065})

    # Only sum if all source values are available.
    # Withheld or missing values remain missing.
    values = (
        values.groupby("gemeinde_bfs", as_index=False)["wert"]
        .agg(lambda series: series.sum(min_count=len(series)))
    )

    column_name = f"sector_{sector}"
    values = values.rename(columns={"wert": column_name})

    employment = employment.merge(
        values,
        on="gemeinde_bfs",
        how="left",
        validate="one_to_one",
    )
    sector_columns.append(column_name)

employment_total = employment[sector_columns].sum(axis=1, min_count=3)

employment["employment_per_100_residents"] = (
    employment_total
    .div(employment["population"].where(employment["population"].gt(0)))
    .mul(100)
)

employment["employment_per_100_residents_year"] = employment_year
employment["employment_source_gemeindestand"] = "01.01.2024"
employment["employment_boundary_adjustment"] = (
    employment["gemeinde_bfs"]
    .eq(1065)
    .map({
        True: "Employment of Honau and Root combined",
        False: "No merger aggregation applied",
    })
)

df_gemeinden = df_gemeinden.merge(
    employment[
        [
            "gemeinde_bfs",
            "employment_per_100_residents",
            "employment_per_100_residents_year",
            "employment_source_gemeindestand",
            "employment_population_gemeindestand",
            "employment_boundary_adjustment",
        ]
    ],
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({
    "variable": "employment_per_100_residents",
    "reference_year": str(employment_year),
})


# Result and completeness

features = [
    "income_median_chf",
    "population_growth_pct",
    "social_assistance_rate_pct",
    "employment_per_100_residents",
    *BFS_INDICATORS,
]

first_columns = [
    "gemeinde_bfs",
    "gemeinde_name",
    "kanton_geo",
    *features,
]

metadata_columns = [
    column for column in df_gemeinden.columns
    if column not in first_columns
]

df_gemeinden = (
    df_gemeinden[first_columns + metadata_columns]
    .sort_values(["kanton_geo", "gemeinde_name"])
    .reset_index(drop=True)
)

print(f"{len(df_gemeinden)} municipalities")
display(df_gemeinden)

print("Reference years used:")
display(pd.DataFrame(year_summary))

print("Missing values per variable:")
display(
    df_gemeinden[features]
    .isna()
    .sum()
    .rename("Municipalities with missing value")
    .to_frame()
)

## Validation

The table is checked before it is saved. Structural errors stop the notebook. The number of municipalities is only a snapshot of the current municipality status; a deviation (e.g. after a merger) is reported as a warning.

In [ ]:
# Snapshot of the municipality status 2025, not a fixed requirement.
EXPECTED_MUNICIPALITIES = 157
# Withheld by the FSO for data protection reasons in small municipalities.
ALLOWED_MISSING = {"social_assistance_rate_pct"}

assert df_gemeinden["gemeinde_bfs"].notna().all(), "Municipality without BFS number"
assert df_gemeinden["gemeinde_bfs"].is_unique, "gemeinde_bfs is not unique"

missing_cantons = set(canton_ranges) - set(df_gemeinden["kanton_geo"])
assert not missing_cantons, f"Cantons without municipalities: {missing_cantons}"

missing = df_gemeinden[features].isna().sum()
unexpected_missing = missing[missing.gt(0) & ~missing.index.isin(ALLOWED_MISSING)]
assert unexpected_missing.empty, f"Unexpected missing values: {unexpected_missing.to_dict()}"

documented = {row["variable"] for row in year_summary}
assert documented == set(features), f"Reference year not documented for: {set(features) ^ documented}"

if len(df_gemeinden) != EXPECTED_MUNICIPALITIES:
    warnings.warn(f"{len(df_gemeinden)} municipalities instead of {EXPECTED_MUNICIPALITIES} (snapshot). "
                  "Check the municipality status of the source files.")

print(f"Validation passed: {len(df_gemeinden)} municipalities in {df_gemeinden['kanton_geo'].nunique()} cantons, "
      f"reference years documented for {len(documented)} indicators, "
      f"missing values only in {sorted(missing[missing.gt(0)].index)}")

In [ ]:
# The municipality status dates remain documented.
# Municipal boundaries are not harmonised by this code.

# Save CSV
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df_gemeinden.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved: {OUTPUT_FILE}")